# 🛒 E-Commerce AI Agent with OpenAI Agents SDK
### RAG-powered product recommendations + Industry-Standard Testing

**Architecture:**
- 📦 Data ingestion via Pandas
- 🔍 RAG with FAISS vector store + OpenAI Embeddings
- 🤖 Multi-agent system using `openai-agents` SDK
- 🧪 Industry-standard tests: Golden Dataset, Curated Q&A, Unit, Hallucination


## 1. 📦 Install Dependencies

In [1]:
!pip install openai openai-agents pandas faiss-cpu numpy tiktoken colorama tabulate pytest pytest-asyncio nest_asyncio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.0/413.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 7.2 MB/s eta 0:00:00


## 2. ⚙️ Imports & Configuration

In [3]:
import os
import json
import asyncio
import numpy as np
import pandas as pd
import faiss
import warnings
import nest_asyncio
from typing import Optional
from openai import OpenAI
from agents import Agent, Runner, function_tool, RunContextWrapper
from dataclasses import dataclass, field
from tabulate import tabulate
from colorama import Fore, Style, init as colorama_init

warnings.filterwarnings('ignore')
nest_asyncio.apply()
colorama_init(autoreset=True)

# ── API Key ──────────────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = ""   # ← replace
client = OpenAI()

EMBED_MODEL   = "text-embedding-3-small"
CHAT_MODEL    = "gpt-4o-mini"
TOP_K         = 5
EMBED_DIM     = 1536

print("✅ Imports OK")

✅ Imports OK


## 3. 📂 Load & Preprocess Dataset

In [5]:
# ── Load your CSV ─────────────────────────────────────────────────────
CSV_PATH = "/content/walmart-products.xlsx"   # ← replace with your actual file path

REQUIRED_COLS = [
    "final_price", "currency", "specifications", "image_urls",
    "available_for_delivery", "available_for_pickup", "brand",
    "review_count", "description", "product_id", "product_name",
    "category_name", "root_category_name", "main_image", "rating",
    "unit_price", "colors", "seller", "other_attributes",
    "customer_reviews", "ingredients"
]

# ── Demo data (used when CSV is not found) ───────────────────────────
DEMO_ROWS = [
    {"product_id": "P001", "product_name": "Organic Green Tea", "brand": "TeaLeaf",
     "category_name": "Tea", "root_category_name": "Beverages",
     "description": "Premium organic green tea sourced from high-altitude farms. Rich in antioxidants.",
     "final_price": 12.99, "unit_price": 12.99, "currency": "USD",
     "rating": 4.7, "review_count": 320,
     "available_for_delivery": True, "available_for_pickup": False,
     "colors": "Green", "seller": "TeaLeaf Store", "specifications": "100g, Loose Leaf",
     "ingredients": "100% Organic Green Tea Leaves",
     "other_attributes": "Vegan, Non-GMO",
     "image_urls": "https://example.com/img1.jpg",
     "main_image": "https://example.com/img1.jpg",
     "customer_reviews": "Great taste! Very fresh. Excellent quality."},
    {"product_id": "P002", "product_name": "Wireless Noise-Cancelling Headphones", "brand": "SoundMax",
     "category_name": "Headphones", "root_category_name": "Electronics",
     "description": "Over-ear headphones with active noise cancellation and 30-hour battery life.",
     "final_price": 149.99, "unit_price": 149.99, "currency": "USD",
     "rating": 4.5, "review_count": 1200,
     "available_for_delivery": True, "available_for_pickup": True,
     "colors": "Black, White, Blue", "seller": "SoundMax Official", "specifications": "Bluetooth 5.0, 30hr battery",
     "ingredients": "",
     "other_attributes": "Foldable, Built-in mic",
     "image_urls": "https://example.com/img2.jpg",
     "main_image": "https://example.com/img2.jpg",
     "customer_reviews": "Amazing sound quality. Very comfortable."},
    {"product_id": "P003", "product_name": "Yoga Mat Pro", "brand": "FitZone",
     "category_name": "Fitness", "root_category_name": "Sports & Outdoors",
     "description": "Non-slip 6mm thick yoga mat with alignment lines. Eco-friendly TPE material.",
     "final_price": 34.99, "unit_price": 34.99, "currency": "USD",
     "rating": 4.6, "review_count": 850,
     "available_for_delivery": True, "available_for_pickup": False,
     "colors": "Purple, Blue, Pink", "seller": "FitZone Sports", "specifications": "183x61cm, 6mm thick",
     "ingredients": "",
     "other_attributes": "Eco-friendly, Non-slip",
     "image_urls": "https://example.com/img3.jpg",
     "main_image": "https://example.com/img3.jpg",
     "customer_reviews": "Great grip. Perfect thickness for joints."},
    {"product_id": "P004", "product_name": "Stainless Steel Water Bottle", "brand": "HydroMax",
     "category_name": "Bottles", "root_category_name": "Kitchen & Dining",
     "description": "Double-walled insulated bottle. Keeps drinks cold 24hrs or hot 12hrs. BPA-free.",
     "final_price": 24.99, "unit_price": 24.99, "currency": "USD",
     "rating": 4.8, "review_count": 2100,
     "available_for_delivery": True, "available_for_pickup": True,
     "colors": "Silver, Black, Red", "seller": "HydroMax", "specifications": "750ml, BPA-free",
     "ingredients": "",
     "other_attributes": "Leak-proof, Dishwasher safe",
     "image_urls": "https://example.com/img4.jpg",
     "main_image": "https://example.com/img4.jpg",
     "customer_reviews": "Keeps water cold all day! Great seal."},
    {"product_id": "P005", "product_name": "Vitamin C Serum", "brand": "GlowSkin",
     "category_name": "Skincare", "root_category_name": "Beauty & Personal Care",
     "description": "20% Vitamin C face serum with hyaluronic acid. Brightens skin and reduces dark spots.",
     "final_price": 28.99, "unit_price": 28.99, "currency": "USD",
     "rating": 4.4, "review_count": 560,
     "available_for_delivery": True, "available_for_pickup": False,
     "colors": "N/A", "seller": "GlowSkin Beauty", "specifications": "30ml, Glass bottle",
     "ingredients": "Ascorbic Acid, Hyaluronic Acid, Vitamin E, Ferulic Acid",
     "other_attributes": "Cruelty-free, Vegan",
     "image_urls": "https://example.com/img5.jpg",
     "main_image": "https://example.com/img5.jpg",
     "customer_reviews": "Skin looks brighter. Reduced my dark spots significantly."},
]

try:
    df = pd.read_excel(CSV_PATH)
    # Fill missing required columns with empty string
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = ""
    print(f"✅ Loaded CSV: {len(df)} rows")
except FileNotFoundError:
    print(f"⚠️  '{CSV_PATH}' not found — using demo dataset (5 products)")
    df = pd.DataFrame(DEMO_ROWS, columns=REQUIRED_COLS)

# Clean up
df.fillna("", inplace=True)
df["rating"]       = pd.to_numeric(df["rating"],       errors="coerce").fillna(0)
df["final_price"]  = pd.to_numeric(df["final_price"],  errors="coerce").fillna(0)
df["review_count"] = pd.to_numeric(df["review_count"], errors="coerce").fillna(0)

print(f"\nDataset shape: {df.shape}")
df.head(2)

✅ Loaded CSV: 1000 rows

Dataset shape: (1000, 21)


,final_price,currency,specifications,image_urls,available_for_delivery,available_for_pickup,brand,review_count,description,product_id,...,category_name,root_category_name,main_image,rating,unit_price,colors,seller,other_attributes,customer_reviews,ingredients
0,22.90,USD,"[{""name"":""Brand"",""value"":""Laura Mercier""},{""na...","[""https://i5.walmartimages.com/seo/Laura-Merci...",True,False,Laura Mercier,6,Laura Mercier Caviar Stick Eye Color Sugar Fro...,173530386,...,Eye Shadow Stick,Beauty,"""https://i5.walmartimages.com/seo/Laura-Mercie...",4.0,22.9,"[""Sugar Frost"",""Tuxedo""]",Wal███t.c███,"[{""name"":""Instructions"",""value"":""Apply directl...","[{""name"":""Jac███"",""rating"":5,""review"":""My only...","Cyclopentasiloxane, trimethylsiloxysilicate, s..."
1,47.88,USD,"[{""name"":""Brand"",""value"":""Exultantex""},{""name""...","[""https://i5.walmartimages.com/seo/Exultantex-...",True,False,Exultantex,58,✨Soft triple weave fabric with a velvet-lik...,430528189,...,Blackout Curtains,Home,"""https://i5.walmartimages.com/seo/Exultantex-G...",4.6,,"[""Black"",""Blue"",""Green"",""Gray"",""Natural(Ivory)...",Exu███nte███ome███,"[{""name"":""Fabric Care Instructions"",""value"":""M...","[{""name"":""Dana"",""rating"":5,""review"":""I love th...",


## 4. 🔍 Build RAG — Embed Products & Create FAISS Index

In [6]:
def build_product_doc(row: pd.Series) -> str:
    """Convert a product row into a rich text document for embedding."""
    parts = [
        f"Product: {row.product_name}",
        f"Brand: {row.brand}",
        f"Category: {row.category_name} > {row.root_category_name}",
        f"Description: {row.description}",
        f"Price: {row.final_price} {row.currency}",
        f"Rating: {row.rating}/5 ({int(row.review_count)} reviews)",
        f"Colors: {row.colors}",
        f"Specs: {row.specifications}",
        f"Seller: {row.seller}",
    ]
    if str(row.ingredients).strip():
        parts.append(f"Ingredients: {row.ingredients}")
    if str(row.other_attributes).strip():
        parts.append(f"Attributes: {row.other_attributes}")
    if str(row.customer_reviews).strip():
        reviews_snip = str(row.customer_reviews)[:200]
        parts.append(f"Customer Reviews: {reviews_snip}")
    return "\n".join(parts)


def get_embeddings(texts: list[str], batch_size: int = 100) -> np.ndarray:
    """Embed texts in batches; returns (N, EMBED_DIM) float32 array."""
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp  = client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_vecs.extend([e.embedding for e in resp.data])
    return np.array(all_vecs, dtype="float32")


print("Building product documents …")
df["_doc"] = df.apply(build_product_doc, axis=1)
docs       = df["_doc"].tolist()

print(f"Embedding {len(docs)} products (this may take a moment) …")
embeddings = get_embeddings(docs)

# FAISS index (L2)
index = faiss.IndexFlatL2(EMBED_DIM)
index.add(embeddings)

print(f"✅ FAISS index built — {index.ntotal} vectors ({EMBED_DIM}d)")

Building product documents …
Embedding 1000 products (this may take a moment) …
✅ FAISS index built — 1000 vectors (1536d)


## 5. 🛠️ RAG Retrieval Utility

In [7]:
def retrieve_products(
    query: str,
    top_k: int = TOP_K,
    min_rating: float = 0.0,
    max_price: float = float("inf"),
    category: str = "",
) -> list[dict]:
    """Semantic search over the FAISS index with optional metadata filters."""
    q_vec = get_embeddings([query])  # (1, dim)
    _, indices = index.search(q_vec, top_k * 3)   # over-fetch then filter

    results = []
    for idx in indices[0]:
        if idx == -1 or idx >= len(df):
            continue
        row = df.iloc[idx]
        if row.rating < min_rating:
            continue
        if row.final_price > max_price:
            continue
        if category and category.lower() not in str(row.category_name).lower():
            continue
        results.append({
            "product_id":    row.product_id,
            "product_name":  row.product_name,
            "brand":         row.brand,
            "category":      row.category_name,
            "price":         row.final_price,
            "currency":      row.currency,
            "rating":        row.rating,
            "review_count":  int(row.review_count),
            "description":   row.description,
            "colors":        row.colors,
            "seller":        row.seller,
            "specifications":row.specifications,
            "ingredients":   row.ingredients,
            "delivery":      row.available_for_delivery,
            "pickup":        row.available_for_pickup,
            "doc_snippet":   row._doc[:400],
        })
        if len(results) >= top_k:
            break
    return results


# Quick sanity check
test_results = retrieve_products("good headphones for music")
print(f"Test query returned {len(test_results)} results")
for r in test_results:
    print(f"  • {r['product_name']} ({r['brand']}) — ${r['price']} ⭐{r['rating']}")

Test query returned 5 results
  • Snailax Vibration Foot Massager with Heat, Adjustable Foot Massager Machine with App Control, Gifts (Snailax) — $69.99 ⭐4.5
  • AMITOFO Compression Socks for Women Wide Calf 3 Pairs Men Circulation 20-30mmHg Plus Size Knee High Support Stockings for Medical | Circulation | Nurses | Running | Travel,Black 4XL (AMITOFO) — $17.99 ⭐4.3
  • HOBIBEAR Barefoot Minimalist Shoes Womens Mens | Zero Drop | Wide Width Fashion Sneaker Wine red,7 Wide Women/6 Wide Men (Hobibear) — $28.99 ⭐4.7
  • Ecetana Mens Womens Water Shoes Quick Dry Barefoot Walking Beach Shoes (Ecetana) — $23.99 ⭐4.2
  • Good Choice Womens Kimora Faux Leather Bow Peep-Toe Heels (Good Choice) — $31.99 ⭐4.3


## 6. 🤖 Define Agent Tools

In [8]:
@function_tool
def search_products(
    query: str,
    top_k: int = 5,
    min_rating: float = 0.0,
    max_price: float = 9999.0,
    category: str = ""
) -> str:
    """
    Search for products using natural language.
    Returns a JSON list of matching products.

    Args:
        query:      Natural language product query.
        top_k:      Max number of results (1-10).
        min_rating: Minimum product rating (0-5).
        max_price:  Maximum price to include.
        category:   Filter by category keyword (optional).
    """
    results = retrieve_products(query, top_k, min_rating, max_price, category)
    if not results:
        return json.dumps({"status": "no_results", "message": "No matching products found."})
    return json.dumps({"status": "ok", "count": len(results), "products": results}, indent=2)


@function_tool
def get_product_details(product_id: str) -> str:
    """
    Retrieve full details for a specific product by its product_id.

    Args:
        product_id: The unique product identifier.
    """
    row = df[df["product_id"].astype(str) == str(product_id)]
    if row.empty:
        return json.dumps({"error": f"Product '{product_id}' not found."})
    r = row.iloc[0].to_dict()
    r.pop("_doc", None)
    return json.dumps(r, indent=2, default=str)


@function_tool
def compare_products(product_ids: list[str]) -> str:
    """
    Compare multiple products side-by-side on price, rating, and features.

    Args:
        product_ids: List of 2-4 product_id strings to compare.
    """
    rows = []
    for pid in product_ids[:4]:
        row = df[df["product_id"].astype(str) == str(pid)]
        if not row.empty:
            r = row.iloc[0]
            rows.append({
                "product_id":   r.product_id,
                "product_name": r.product_name,
                "brand":        r.brand,
                "price":        r.final_price,
                "currency":     r.currency,
                "rating":       r.rating,
                "review_count": int(r.review_count),
                "specifications": str(r.specifications)[:100],
                "colors":       r.colors,
                "delivery":     r.available_for_delivery,
                "pickup":       r.available_for_pickup,
            })
    if not rows:
        return json.dumps({"error": "No valid product IDs found."})
    return json.dumps({"comparison": rows}, indent=2)


@function_tool
def get_top_rated(category: str = "", top_n: int = 5, min_reviews: int = 50) -> str:
    """
    Return the top-rated products, optionally filtered by category.

    Args:
        category:    Category keyword to filter by (empty = all categories).
        top_n:       Number of products to return.
        min_reviews: Minimum review count for inclusion.
    """
    filtered = df[df["review_count"] >= min_reviews].copy()
    if category:
        filtered = filtered[
            filtered["category_name"].str.contains(category, case=False, na=False)
        ]
    top = filtered.nlargest(top_n, "rating")[
        ["product_id","product_name","brand","category_name","rating","review_count","final_price","currency"]
    ].to_dict(orient="records")
    if not top:
        return json.dumps({"status": "no_results", "message": "No products match the criteria."})
    return json.dumps({"status": "ok", "top_rated": top}, indent=2)


@function_tool
def check_availability(product_id: str) -> str:
    """
    Check delivery and pickup availability for a product.

    Args:
        product_id: The unique product identifier.
    """
    row = df[df["product_id"].astype(str) == str(product_id)]
    if row.empty:
        return json.dumps({"error": f"Product '{product_id}' not found."})
    r = row.iloc[0]
    return json.dumps({
        "product_id":            r.product_id,
        "product_name":          r.product_name,
        "available_for_delivery": bool(r.available_for_delivery),
        "available_for_pickup":  bool(r.available_for_pickup),
        "seller":                r.seller,
    })


print("✅ Tools defined:", [
    search_products.name,
    get_product_details.name,
    compare_products.name,
    get_top_rated.name,
    check_availability.name
])

✅ Tools defined: ['search_products', 'get_product_details', 'compare_products', 'get_top_rated', 'check_availability']


## 7. 🤖 Create Specialist Agents

In [9]:
# ── 1. Recommendation Agent ───────────────────────────────────────────
recommendation_agent = Agent(
    name="RecommendationAgent",
    model=CHAT_MODEL,
    instructions="""
You are a friendly, knowledgeable e-commerce recommendation assistant.
Your goal is to help users discover products that match their needs.

Guidelines:
- Always use search_products to find relevant items before recommending.
- Present recommendations clearly with name, price, rating, and a short reason.
- If the user mentions a budget, pass it as max_price.
- If they mention a minimum quality expectation, use min_rating.
- Be conversational, warm, and helpful.
- Never invent product details — only use tool results.
""",
    tools=[search_products, get_product_details, get_top_rated],
)

# ── 2. Product Details Agent ──────────────────────────────────────────
product_details_agent = Agent(
    name="ProductDetailsAgent",
    model=CHAT_MODEL,
    instructions="""
You are a product information specialist for an e-commerce platform.
Provide accurate, detailed information about specific products.

Guidelines:
- Always call get_product_details first before answering questions about a product.
- For availability questions use check_availability.
- Summarise specifications in a readable way.
- Be precise and factual — do not guess or fabricate specs.
""",
    tools=[get_product_details, check_availability],
)

# ── 3. Comparison Agent ───────────────────────────────────────────────
comparison_agent = Agent(
    name="ComparisonAgent",
    model=CHAT_MODEL,
    instructions="""
You are a product comparison expert helping shoppers make informed decisions.

Guidelines:
- When asked to compare, first search for both/all products to get their IDs.
- Then call compare_products with those IDs.
- Present a clear comparison table (markdown) highlighting:
  price, rating, key specs, pros and cons.
- Conclude with a concise recommendation based on use-case.
- Never fabricate specs or ratings.
""",
    tools=[search_products, compare_products],
)

# ── 4. Orchestrator (Triage) Agent ───────────────────────────────────
orchestrator_agent = Agent(
    name="EcommerceOrchestrator",
    model=CHAT_MODEL,
    instructions="""
You are the main e-commerce assistant that routes user requests.

Routing rules:
- 'recommend', 'suggest', 'best', 'top', 'looking for' → use RecommendationAgent
- 'details', 'specs', 'ingredients', 'tell me about', 'available', 'delivery' → use ProductDetailsAgent
- 'compare', 'vs', 'difference', 'which is better' → use ComparisonAgent
- For all other queries, handle directly using the available tools.

Always be helpful, concise, and friendly.
""",
    tools=[search_products, get_product_details, get_top_rated, check_availability, compare_products],
    handoffs=[recommendation_agent, product_details_agent, comparison_agent],
)

print("✅ Agents created: Orchestrator → [Recommendation, ProductDetails, Comparison]")

✅ Agents created: Orchestrator → [Recommendation, ProductDetails, Comparison]


## 8. 💬 Interactive Query Runner

In [10]:
async def ask(query: str, agent: Agent = orchestrator_agent, verbose: bool = True) -> str:
    """Run a single query through the agent pipeline."""
    if verbose:
        print(f"{Fore.CYAN}\n🧑 User: {query}{Style.RESET_ALL}")
    result = await Runner.run(agent, query)
    response = result.final_output
    if verbose:
        print(f"{Fore.GREEN}🤖 Agent: {response}{Style.RESET_ALL}")
    return response


# ── Demo queries ──────────────────────────────────────────────────────
demo_queries = [
    "Recommend me a good product under $50 with high ratings.",
    "What are the ingredients in the Vitamin C Serum?",
    "Compare the headphones and the yoga mat.",
    "Show me the top rated products in fitness.",
]

for q in demo_queries:
    await ask(q)
    print("-" * 70)


🧑 User: Recommend me a good product under $50 with high ratings.
🤖 Agent: Here are some highly-rated products under $50 that you might like:

1. **ChainsHouse Snake Ring**  
   - **Price**: $12.13  
   - **Rating**: 5.0 (50 reviews)  
   - **Description**: Stainless Steel Retro Punk Gothic Jewelry, available in sizes 7-14.

2. **KOOLSOLY Outdoor Sun Hat**  
   - **Price**: $9.99  
   - **Rating**: 5.0 (59 reviews)  
   - **Description**: Wide brim fishing hat with 50+ UPF protection, perfect for outdoor activities.

3. **FaithHeart Nordic Viking Ring**  
   - **Price**: $15.95  
   - **Rating**: 4.9 (150 reviews)  
   - **Description**: Solid black stainless steel jewelry featuring a pirate compass design.

4. **Leading Lady Microfiber Bralette**  
   - **Price**: $13.50  
   - **Rating**: 4.9 (50 reviews)  
   - **Description**: Wireless lace trim bralette, comfortable and stylish.

If you need more information on any of these products, just let me know!
-----------------------------

MaxTurnsExceeded: Max turns (10) exceeded

---
# 🧪 Industry-Standard Testing Suite
### Tests: Golden Dataset · Curated Q&A · Unit (Prompt) · Hallucination

## 9. 🥇 Golden Dataset — Ground-Truth Retrieval Precision

In [11]:
"""
Golden Dataset Test
────────────────────
A curated set of (query, expected_product_id) pairs.
Measures Precision@K — how often the expected product
appears in the top-K retrieved results.
"""

GOLDEN_DATASET = [
    {"query": "organic green tea antioxidants",          "expected_id": "P001", "expected_name": "Organic Green Tea"},
    {"query": "noise cancelling bluetooth headphones",   "expected_id": "P002", "expected_name": "Wireless Noise-Cancelling Headphones"},
    {"query": "non-slip yoga mat eco friendly",          "expected_id": "P003", "expected_name": "Yoga Mat Pro"},
    {"query": "insulated water bottle BPA free",         "expected_id": "P004", "expected_name": "Stainless Steel Water Bottle"},
    {"query": "vitamin c face serum skin brightening",   "expected_id": "P005", "expected_name": "Vitamin C Serum"},
    {"query": "exercise mat for yoga and pilates",       "expected_id": "P003", "expected_name": "Yoga Mat Pro"},
    {"query": "hydration bottle for gym",                "expected_id": "P004", "expected_name": "Stainless Steel Water Bottle"},
    {"query": "skincare serum with hyaluronic acid",     "expected_id": "P005", "expected_name": "Vitamin C Serum"},
]

def run_golden_dataset_tests(
    dataset: list[dict],
    top_k: int = TOP_K,
) -> dict:
    """Evaluate retrieval precision against golden dataset."""
    hits, misses, rows = 0, 0, []

    for item in dataset:
        results    = retrieve_products(item["query"], top_k=top_k)
        ret_ids    = [str(r["product_id"]) for r in results]
        ret_names  = [r["product_name"]    for r in results]
        hit        = item["expected_id"] in ret_ids
        hits      += int(hit)
        misses    += int(not hit)
        rows.append([
            item["query"][:45],
            item["expected_name"][:30],
            "✅ HIT" if hit else "❌ MISS",
            ret_names[0][:30] if ret_names else "—"
        ])

    precision = hits / len(dataset)
    print("\n" + "="*70)
    print("🥇  GOLDEN DATASET — Retrieval Precision@K")
    print("="*70)
    print(tabulate(
        rows,
        headers=["Query", "Expected Product", "Result", "Top Returned"],
        tablefmt="fancy_grid"
    ))
    colour = Fore.GREEN if precision >= 0.8 else Fore.YELLOW if precision >= 0.6 else Fore.RED
    print(f"\n{colour}Precision@{top_k}: {precision:.1%}  ({hits}/{len(dataset)} hits){Style.RESET_ALL}")
    return {"precision": precision, "hits": hits, "total": len(dataset)}

golden_results = run_golden_dataset_tests(GOLDEN_DATASET)


🥇  GOLDEN DATASET — Retrieval Precision@K
╒═══════════════════════════════════════╤════════════════════════════════╤══════════╤════════════════════════════════╕
│ Query                                 │ Expected Product               │ Result   │ Top Returned                   │
╞═══════════════════════════════════════╪════════════════════════════════╪══════════╪════════════════════════════════╡
│ organic green tea antioxidants        │ Organic Green Tea              │ ❌ MISS  │ Organic Maqui Berry Powder, 2  │
├───────────────────────────────────────┼────────────────────────────────┼──────────┼────────────────────────────────┤
│ noise cancelling bluetooth headphones │ Wireless Noise-Cancelling Head │ ❌ MISS  │ ALANDA Cooling Pillowcases, 2  │
├───────────────────────────────────────┼────────────────────────────────┼──────────┼────────────────────────────────┤
│ non-slip yoga mat eco friendly        │ Yoga Mat Pro                   │ ❌ MISS  │ SIXHOME Outdoor Mat Non Slip D │
├───────

## 10. 📋 Curated Q&A Evaluation — Answer Quality Scoring

In [12]:
"""
Curated Q&A Evaluation
────────────────────────
Each test case has a question, expected keywords, and forbidden keywords.
An LLM judge scores each answer 1-5, then we also do keyword matching.
"""

CURATED_QA = [
    {
        "id": "QA-01",
        "question": "What are the ingredients in the Vitamin C Serum?",
        "expected_keywords": ["ascorbic acid", "hyaluronic acid", "vitamin e", "ferulic acid"],
        "forbidden_keywords": ["don't know", "not available", "no information"],
        "min_score": 4,
    },
    {
        "id": "QA-02",
        "question": "Recommend a product under $30 with a rating above 4.5.",
        "expected_keywords": ["green tea", "yoga mat"],   # both fit criteria
        "forbidden_keywords": ["headphones", "serum"],    # both cost more or rate lower
        "min_score": 3,
    },
    {
        "id": "QA-03",
        "question": "Is the stainless steel water bottle available for pickup?",
        "expected_keywords": ["pickup", "yes", "available"],
        "forbidden_keywords": ["not available for pickup", "no pickup"],
        "min_score": 4,
    },
    {
        "id": "QA-04",
        "question": "Which brand makes the noise-cancelling headphones?",
        "expected_keywords": ["soundmax"],
        "forbidden_keywords": ["hydromax", "fitzone", "glowskin", "tealeaf"],
        "min_score": 4,
    },
    {
        "id": "QA-05",
        "question": "Show me the best fitness products.",
        "expected_keywords": ["yoga", "mat", "fitzone"],
        "forbidden_keywords": ["no products", "cannot find"],
        "min_score": 3,
    },
]


def llm_score_answer(question: str, answer: str) -> tuple[int, str]:
    """Ask GPT-4o-mini to score an answer 1-5 and return (score, reason)."""
    prompt = f"""You are an evaluator for an e-commerce Q&A system.
Score the answer below on a scale of 1-5:
5 = Excellent: accurate, complete, helpful
4 = Good: mostly accurate, minor gaps
3 = Acceptable: partially correct
2 = Poor: significant gaps or errors
1 = Fail: wrong, unhelpful, or hallucinated

Question: {question}
Answer: {answer}

Respond ONLY with JSON: {{"score": <int>, "reason": "<one sentence>"}}"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=100,
    )
    text = resp.choices[0].message.content.strip()
    try:
        data = json.loads(text)
        return int(data["score"]), data.get("reason", "")
    except Exception:
        return 3, "Parse error"


async def run_curated_qa_tests(qa_set: list[dict]) -> dict:
    """Run all curated Q&A tests and report results."""
    rows, passed, total = [], 0, len(qa_set)

    for item in qa_set:
        answer = await ask(item["question"], verbose=False)
        ans_lower = answer.lower()

        # Keyword checks
        kw_hits     = sum(1 for kw in item["expected_keywords"]  if kw.lower() in ans_lower)
        kw_forbidden = sum(1 for kw in item["forbidden_keywords"] if kw.lower() in ans_lower)

        # LLM judge
        score, reason = llm_score_answer(item["question"], answer)

        ok = (score >= item["min_score"]) and (kw_forbidden == 0)
        passed += int(ok)

        rows.append([
            item["id"],
            item["question"][:40],
            f"{score}/5",
            f"{kw_hits}/{len(item['expected_keywords'])}",
            f"{kw_forbidden} ❌" if kw_forbidden else "0 ✅",
            "✅ PASS" if ok else "❌ FAIL",
            reason[:50]
        ])

    acc = passed / total
    print("\n" + "="*90)
    print("📋  CURATED Q&A EVALUATION")
    print("="*90)
    print(tabulate(
        rows,
        headers=["ID","Question","Score","KW Hits","Forbidden","Status","Reason"],
        tablefmt="fancy_grid"
    ))
    colour = Fore.GREEN if acc >= 0.8 else Fore.YELLOW if acc >= 0.6 else Fore.RED
    print(f"\n{colour}Accuracy: {acc:.1%}  ({passed}/{total} passed){Style.RESET_ALL}")
    return {"accuracy": acc, "passed": passed, "total": total}

qa_results = await run_curated_qa_tests(CURATED_QA)


📋  CURATED Q&A EVALUATION
╒═══════╤══════════════════════════════════════════╤═════════╤═══════════╤═════════════╤══════════╤════════════════════════════════════════════════════╕
│ ID    │ Question                                 │ Score   │ KW Hits   │ Forbidden   │ Status   │ Reason                                             │
╞═══════╪══════════════════════════════════════════╪═════════╪═══════════╪═════════════╪══════════╪════════════════════════════════════════════════════╡
│ QA-01 │ What are the ingredients in the Vitamin  │ 1/5     │ 0/4       │ 0 ✅        │ ❌ FAIL  │ The answer does not provide any information about  │
├───────┼──────────────────────────────────────────┼─────────┼───────────┼─────────────┼──────────┼────────────────────────────────────────────────────┤
│ QA-02 │ Recommend a product under $30 with a rat │ 5/5     │ 0/2       │ 0 ✅        │ ✅ PASS  │ The answer provides accurate, complete, and helpfu │
├───────┼──────────────────────────────────────────┼───────

## 11. 🔬 Unit Testing — Prompt & Tool Behaviour

In [13]:
"""
Unit Tests for Prompts & Tools
───────────────────────────────
Tests are grouped into:
  A) Tool unit tests  (pure function checks)
  B) Agent prompt tests (assert response patterns)
"""
import unittest

class TestTools(unittest.TestCase):

    # ── Tool: search_products ─────────────────────────────────────────
    def test_search_returns_results(self):
        """search_products returns at least 1 result for a generic query."""
        raw = search_products.on_invoke_tool(None, json.dumps({"query": "product"}))
        data = json.loads(raw)
        self.assertEqual(data["status"], "ok")
        self.assertGreater(data["count"], 0)

    def test_search_max_price_filter(self):
        """All returned products respect max_price filter."""
        raw = search_products.on_invoke_tool(
            None, json.dumps({"query": "product", "max_price": 20.0})
        )
        data = json.loads(raw)
        if data["status"] == "ok":
            for p in data["products"]:
                self.assertLessEqual(p["price"], 20.0)

    def test_search_min_rating_filter(self):
        """All returned products meet min_rating."""
        raw = search_products.on_invoke_tool(
            None, json.dumps({"query": "product", "min_rating": 4.5})
        )
        data = json.loads(raw)
        if data["status"] == "ok":
            for p in data["products"]:
                self.assertGreaterEqual(p["rating"], 4.5)

    def test_search_top_k_limit(self):
        """Number of results does not exceed top_k."""
        raw = search_products.on_invoke_tool(
            None, json.dumps({"query": "product", "top_k": 2})
        )
        data = json.loads(raw)
        if data["status"] == "ok":
            self.assertLessEqual(data["count"], 2)

    # ── Tool: get_product_details ─────────────────────────────────────
    def test_get_product_details_valid_id(self):
        """Returns product data for a known product_id."""
        raw  = get_product_details.on_invoke_tool(None, json.dumps({"product_id": "P001"}))
        data = json.loads(raw)
        self.assertNotIn("error", data)
        self.assertEqual(str(data["product_id"]), "P001")

    def test_get_product_details_invalid_id(self):
        """Returns error dict for unknown product_id."""
        raw  = get_product_details.on_invoke_tool(None, json.dumps({"product_id": "NONEXISTENT_XYZ"}))
        data = json.loads(raw)
        self.assertIn("error", data)

    # ── Tool: compare_products ────────────────────────────────────────
    def test_compare_products_returns_all(self):
        """Comparison includes all requested IDs."""
        raw  = compare_products.on_invoke_tool(None, json.dumps({"product_ids": ["P001", "P002"]}))
        data = json.loads(raw)
        self.assertIn("comparison", data)
        self.assertEqual(len(data["comparison"]), 2)

    def test_compare_capped_at_four(self):
        """Comparison caps results at 4 products max."""
        raw  = compare_products.on_invoke_tool(
            None, json.dumps({"product_ids": ["P001","P002","P003","P004","P005"]})
        )
        data = json.loads(raw)
        self.assertLessEqual(len(data.get("comparison", [])), 4)

    # ── Tool: get_top_rated ───────────────────────────────────────────
    def test_top_rated_returns_sorted(self):
        """Top-rated results are in descending rating order."""
        raw  = get_top_rated.on_invoke_tool(None, json.dumps({"min_reviews": 0}))
        data = json.loads(raw)
        if data["status"] == "ok":
            ratings = [r["rating"] for r in data["top_rated"]]
            self.assertEqual(ratings, sorted(ratings, reverse=True))

    # ── Tool: check_availability ──────────────────────────────────────
    def test_check_availability_keys_present(self):
        """Availability response contains expected keys."""
        raw  = check_availability.on_invoke_tool(None, json.dumps({"product_id": "P004"}))
        data = json.loads(raw)
        for key in ["product_id","available_for_delivery","available_for_pickup"]:
            self.assertIn(key, data)

    # ── Retrieval quality ─────────────────────────────────────────────
    def test_retrieve_product_by_exact_name(self):
        """Exact product name query returns that product at rank 1."""
        results = retrieve_products("Organic Green Tea", top_k=1)
        self.assertTrue(len(results) > 0)
        self.assertIn("Green Tea", results[0]["product_name"])

    def test_retrieve_empty_query_no_crash(self):
        """Empty-ish query does not raise an exception."""
        try:
            retrieve_products("     ")
        except Exception as e:
            self.fail(f"retrieve_products raised {e} on whitespace query")


# Run tests
loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestTools)
runner = unittest.TextTestRunner(verbosity=2)
unit_result = runner.run(suite)

n_run  = unit_result.testsRun
n_fail = len(unit_result.failures) + len(unit_result.errors)
n_pass = n_run - n_fail
colour = Fore.GREEN if n_fail == 0 else Fore.RED
print(f"\n{colour}Unit Tests: {n_pass}/{n_run} passed{Style.RESET_ALL}")

test_check_availability_keys_present (__main__.TestTools.test_check_availability_keys_present)
Availability response contains expected keys. ... ERROR
test_compare_capped_at_four (__main__.TestTools.test_compare_capped_at_four)
Comparison caps results at 4 products max. ... ERROR
test_compare_products_returns_all (__main__.TestTools.test_compare_products_returns_all)
Comparison includes all requested IDs. ... ERROR
test_get_product_details_invalid_id (__main__.TestTools.test_get_product_details_invalid_id)
Returns error dict for unknown product_id. ... ERROR
test_get_product_details_valid_id (__main__.TestTools.test_get_product_details_valid_id)
Returns product data for a known product_id. ... ERROR
test_retrieve_empty_query_no_crash (__main__.TestTools.test_retrieve_empty_query_no_crash)
Empty-ish query does not raise an exception. ... ok
test_retrieve_product_by_exact_name (__main__.TestTools.test_retrieve_product_by_exact_name)
Exact product name query returns that product at rank 1


Unit Tests: 1/12 passed


## 12. 🕵️ Hallucination Testing

In [14]:
"""
Hallucination Tests
─────────────────────
Three categories:
  1. Out-of-scope queries   — agent should decline / say unknown
  2. Fake product queries   — agent must NOT invent product details
  3. Factual grounding      — claims must be verifiable against dataset
"""

HALLUCINATION_TESTS = [
    # Category 1 — Out-of-scope
    {
        "id": "HAL-01",
        "category": "out_of_scope",
        "query": "What is the capital of France?",
        "must_not_contain": ["paris"],   # should redirect, not just answer
        "must_contain_any": ["product", "shop", "e-commerce", "catalog", "assist", "help"],
        "description": "Off-topic question should be redirected to product domain"
    },
    # Category 2 — Fake products
    {
        "id": "HAL-02",
        "category": "fake_product",
        "query": "Tell me about the SuperDrone X9000 quantum levitation device.",
        "must_not_contain": ["superdrone", "x9000", "quantum levitation"],
        "must_contain_any": ["not found", "don't have", "no product", "unable", "couldn't find"],
        "description": "Agent must not invent details for non-existent products"
    },
    {
        "id": "HAL-03",
        "category": "fake_product",
        "query": "What's the price of the NanoBot Pro 3000 smartwatch?",
        "must_not_contain": ["nanobot", "$299", "$199"],  # fabricated prices
        "must_contain_any": ["not found", "don't have", "no product", "unable", "couldn't find", "no match"],
        "description": "Agent must not fabricate prices for unknown products"
    },
    # Category 3 — Factual grounding
    {
        "id": "HAL-04",
        "category": "factual_grounding",
        "query": "What is the price of the Organic Green Tea?",
        "must_contain_any": ["12.99", "$12"],
        "must_not_contain": ["$5", "$50", "$100"],
        "description": "Price must match dataset value ($12.99)"
    },
    {
        "id": "HAL-05",
        "category": "factual_grounding",
        "query": "What brand makes the noise-cancelling headphones?",
        "must_contain_any": ["soundmax"],
        "must_not_contain": ["sony", "bose", "apple", "hydromax", "tealeaf"],
        "description": "Brand must be SoundMax, not a hallucinated brand"
    },
    {
        "id": "HAL-06",
        "category": "factual_grounding",
        "query": "How many reviews does the stainless steel water bottle have?",
        "must_contain_any": ["2100", "2,100"],
        "must_not_contain": ["500", "100", "50"],
        "description": "Review count must match dataset (2100)"
    },
]


def detect_hallucination(response: str, test: dict) -> tuple[bool, list[str], list[str]]:
    """Return (passed, found_forbidden, missing_required)."""
    resp_lower    = response.lower()
    found_bad     = [kw for kw in test["must_not_contain"] if kw.lower() in resp_lower]
    found_good    = [kw for kw in test["must_contain_any"]  if kw.lower() in resp_lower]
    missing_good  = [] if found_good else test["must_contain_any"]
    passed        = (len(found_bad) == 0) and bool(found_good)
    return passed, found_bad, missing_good


async def run_hallucination_tests(tests: list[dict]) -> dict:
    """Run hallucination tests and report."""
    rows, passed, total = [], 0, len(tests)

    for test in tests:
        answer             = await ask(test["query"], verbose=False)
        ok, bad, missing   = detect_hallucination(answer, test)
        passed            += int(ok)

        issues = []
        if bad:     issues.append(f"Forbidden: {bad}")
        if missing: issues.append(f"Missing: {missing[:2]}")

        rows.append([
            test["id"],
            test["category"],
            test["description"][:45],
            "✅ PASS" if ok else "❌ FAIL",
            "; ".join(issues)[:60] if issues else "—"
        ])

    acc = passed / total
    print("\n" + "="*90)
    print("🕵️  HALLUCINATION TESTS")
    print("="*90)
    print(tabulate(
        rows,
        headers=["ID", "Category", "Description", "Status", "Issues"],
        tablefmt="fancy_grid"
    ))
    colour = Fore.GREEN if acc >= 0.8 else Fore.YELLOW if acc >= 0.6 else Fore.RED
    print(f"\n{colour}Hallucination Pass Rate: {acc:.1%}  ({passed}/{total}){Style.RESET_ALL}")
    return {"pass_rate": acc, "passed": passed, "total": total}

hal_results = await run_hallucination_tests(HALLUCINATION_TESTS)


🕵️  HALLUCINATION TESTS
╒════════╤═══════════════════╤═══════════════════════════════════════════════╤══════════╤═════════════════════════════════════════════════════════════╕
│ ID     │ Category          │ Description                                   │ Status   │ Issues                                                      │
╞════════╪═══════════════════╪═══════════════════════════════════════════════╪══════════╪═════════════════════════════════════════════════════════════╡
│ HAL-01 │ out_of_scope      │ Off-topic question should be redirected to pr │ ❌ FAIL  │ Forbidden: ['paris']                                        │
├────────┼───────────────────┼───────────────────────────────────────────────┼──────────┼─────────────────────────────────────────────────────────────┤
│ HAL-02 │ fake_product      │ Agent must not invent details for non-existen │ ❌ FAIL  │ Forbidden: ['superdrone', 'x9000', 'quantum levitation']    │
├────────┼───────────────────┼───────────────────────────────────

## 13. 📊 Overall Test Report

In [ ]:
def print_final_report(golden, qa, unit_res, hal):
    unit_pass = unit_res.testsRun - len(unit_res.failures) - len(unit_res.errors)
    unit_acc  = unit_pass / max(unit_res.testsRun, 1)

    rows = [
        ["🥇 Golden Dataset (Retrieval Precision@K)",
         f"{golden['precision']:.1%}", f"{golden['hits']}/{golden['total']}",
         "✅" if golden['precision'] >= 0.7 else "⚠️"],
        ["📋 Curated Q&A (Answer Quality)",
         f"{qa['accuracy']:.1%}",      f"{qa['passed']}/{qa['total']}",
         "✅" if qa['accuracy'] >= 0.7 else "⚠️"],
        ["🔬 Unit Tests (Tool & Prompt)",
         f"{unit_acc:.1%}",            f"{unit_pass}/{unit_res.testsRun}",
         "✅" if unit_acc >= 0.9 else "⚠️"],
        ["🕵️  Hallucination Tests",
         f"{hal['pass_rate']:.1%}",    f"{hal['passed']}/{hal['total']}",
         "✅" if hal['pass_rate'] >= 0.8 else "⚠️"],
    ]

    print("\n" + "═"*70)
    print("📊  FINAL EVALUATION REPORT — E-Commerce Agent".center(70))
    print("═"*70)
    print(tabulate(
        rows,
        headers=["Test Suite", "Score", "Passed", "Status"],
        tablefmt="fancy_grid"
    ))

    scores = [golden['precision'], qa['accuracy'], unit_acc, hal['pass_rate']]
    overall = sum(scores) / len(scores)
    colour  = Fore.GREEN if overall >= 0.8 else Fore.YELLOW if overall >= 0.6 else Fore.RED
    print(f"\n{colour}Overall Score: {overall:.1%}{Style.RESET_ALL}")
    print("\nLegend: ✅ ≥70%  ⚠️ <70%")
    print("═"*70)

print_final_report(golden_results, qa_results, unit_result, hal_results)

## 14. 🎛️ Interactive Chat (run any query)

In [ ]:
# Run any custom query through the orchestrator
user_query = "Recommend me a good skincare product with natural ingredients under $40."
response   = await ask(user_query)